# MinHash+LSH 聚类分词器 vs BPE — 端到端实验

**一键跑通：分词器训练 → 数据准备 → GPT-2 训练 × 2 → BPB 对比**

### 使用前
1. 菜单 → **Runtime → Change runtime type → GPU**
2. 推荐选 **A100**（Colab Pro）或 T4（免费）
3. 顺序执行所有 Cell

### 显存 / 参数对照
| GPU | VRAM | train_mb | batch_size | seq_len | iters |
|-----|------|----------|------------|---------|-------|
| T4（免费） | 16 GB | 20 | 8 | 512 | 3 000 |
| V100 | 16 GB | 20 | 8 | 512 | 3 000 |
| A100 40 GB | 40 GB | 200 | 32 | 1024 | 10 000 |
| A100 80 GB | 80 GB | 500 | 64 | 1024 | 10 000 |

In [ ]:
import torch, subprocess

# GPU info
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
print("GPU:", r.stdout.strip() or "none")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

# A100: enable TF32 + BF16
DTYPE = "float32"
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  {name}  {vram:.0f} GB")
    if "A100" in name or "H100" in name:
        torch.set_float32_matmul_precision("high")
        DTYPE = "bfloat16"
        print("  → TF32 + BF16 enabled")
else:
    vram = 0

# Auto-select run parameters
def auto_cfg(vram):
    if vram >= 75:   return dict(train_mb=500, bs=64,  seq=1024, iters=10000)
    if vram >= 35:   return dict(train_mb=200, bs=32,  seq=1024, iters=10000)
    if vram >= 20:   return dict(train_mb=50,  bs=16,  seq=1024, iters=5000)
    if vram >= 10:   return dict(train_mb=20,  bs=8,   seq=512,  iters=3000)
    return           dict(train_mb=5,   bs=4,   seq=256,  iters=500)   # CPU fallback

CFG = auto_cfg(vram)
print("\nConfig:", CFG)

In [ ]:
# Set USE_DRIVE=False to skip (data lost when session ends)
USE_DRIVE = True
import os

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/lp_tok_exp"
    except ModuleNotFoundError:
        print("Not in Colab, using /tmp")
        WORK = "/tmp/lp_tok_exp"
else:
    WORK = "/tmp/lp_tok_exp"

os.makedirs(WORK, exist_ok=True)
print("Work dir:", WORK)

In [ ]:
%pip install -q tiktoken datasets tqdm

In [ ]:
import os, subprocess

REPO = os.path.join(WORK, "lp_tokenizer")

if not os.path.isdir(REPO):
    # ← replace with your actual repo URL
    subprocess.run(["git", "clone",
                    "https://github.com/kelvinfkr/lp_tokenizer", REPO],
                   check=True)
    print("Cloned to", REPO)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout",
                    "claude/clustering-tokenizer-experiment-eVXUR"], check=False)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
    print("Pulled latest in", REPO)

os.chdir(REPO)
print("CWD:", os.getcwd())

In [ ]:
import subprocess, sys

# TinyStories (~2 GB total, streaming so only fetches what's needed)
subprocess.run([sys.executable, "data/download.py", "--dataset", "tinystories"],
               check=True)

In [ ]:
import subprocess, sys, os

def run_script(args, label=""):
    """Run a script and stream its output. Raises on non-zero exit."""
    print(f"{'='*55}\n  {label or ' '.join(args[-1:])}\n{'='*55}")
    proc = subprocess.Popen(args, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (rc={proc.returncode}): {' '.join(args)}")

run_script([sys.executable, "data/prepare_bpe.py"], "prepare_bpe")
print("BPE shards:", sorted(os.listdir("data/bpe"))[:6], "...")

In [ ]:
import subprocess, sys

# run_script() was defined in the previous cell; re-define here in case
# cells are run out of order.
def run_script(args, label=""):
    print(f"{'='*55}\n  {label or ' '.join(args[-1:])}\n{'='*55}")
    proc = subprocess.Popen(args, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (rc={proc.returncode}): {' '.join(args)}")

run_script([
    sys.executable, "data/prepare_clustering.py",
    "--train_mb",  str(CFG["train_mb"]),
    "--vocab_size", "50257",
], "prepare_clustering")

In [ ]:
import subprocess, sys, os, re, time

TRAIN_PY = "llm.c/train_gpt2.py"

def run_training(name, input_glob, val_bin, out_dir, cfg, dtype):
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, "log.txt")

    cmd = [
        sys.executable, TRAIN_PY,
        "--input_bin",              input_glob,
        "--input_val_bin",          val_bin,
        "--output_dir",             out_dir,
        "--model",                  "d12",
        "--batch_size",             str(cfg["bs"]),
        "--sequence_length",        str(cfg["seq"]),
        "--total_batch_size",       "16384",
        "--num_iterations",         str(cfg["iters"]),
        "--learning_rate",          "3e-4",
        "--warmup_iters",           "200",
        "--learning_rate_decay_frac", "0.1",
        "--weight_decay",           "0.1",
        "--grad_clip",              "1.0",
        "--val_loss_every",         "100",
        "--val_max_steps",          "20",
        "--tensorcores",            "1",
        "--overfit_single_batch",   "0",
        "--dtype",                  dtype,
        "--write_tensors",          "0",   # skip saving .bin weights (saves disk)
    ]

    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"  dtype={dtype}  bs={cfg['bs']}  seq={cfg['seq']}  iters={cfg['iters']}")
    print(f"{'='*60}")

    t0 = time.time()
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            logf.write(line)
            # Print every 10th step + all val lines
            m = re.search(r"step\s+(\d+)", line)
            if m and int(m.group(1)) % 500 == 0:
                print(line.rstrip())
            elif "val loss" in line or "ERROR" in line.upper():
                print(line.rstrip())
        proc.wait()

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min  →  {log_path}")
    if proc.returncode != 0:
        raise RuntimeError(f"Training failed (rc={proc.returncode}). Check {log_path}")

print("Helper defined.")

In [ ]:
run_training(
    name       = "BPE",
    input_glob = "data/bpe/train_*.bin",
    val_bin    = "data/bpe/val_0000.bin",
    out_dir    = "experiments/out/bpe",
    cfg        = CFG,
    dtype      = DTYPE,
)

In [ ]:
run_training(
    name       = "Clustering (MinHash+LSH)",
    input_glob = "data/clustering/train_*.bin",
    val_bin    = "data/clustering/val_0000.bin",
    out_dir    = "experiments/out/clustering",
    cfg        = CFG,
    dtype      = DTYPE,
)

In [ ]:
import re, numpy as np

def parse_log(path):
    """Returns list of (step, val_loss) tuples."""
    records, pending_step = [], None
    with open(path) as f:
        for line in f:
            m = re.search(r"step\s+(\d+)", line)
            if m:
                pending_step = int(m.group(1))
            m2 = re.search(r"val loss\s+([\d.]+)", line)
            if m2 and pending_step is not None:
                records.append((pending_step, float(m2.group(1))))
                pending_step = None
    return records

bpe_log = parse_log("experiments/out/bpe/log.txt")
cls_log = parse_log("experiments/out/clustering/log.txt")

print(f"BPE:        {len(bpe_log)} val checkpoints, "
      f"final loss = {bpe_log[-1][1]:.4f}" if bpe_log else "no val records")
print(f"Clustering: {len(cls_log)} val checkpoints, "
      f"final loss = {cls_log[-1][1]:.4f}" if cls_log else "no val records")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# avg bytes per token:
#   BPE (GPT-2) ~ 3.6 on English text
#   Clustering: load from tokenizer to compute precisely
BPE_BPT = 3.6

def get_cls_bpt(tok_path="data/clustering/tokenizer.bin"):
    try:
        import sys; sys.path.insert(0, ".")
        from tokenizer.clustering_tokenizer import ClusteringTokenizer
        tok = ClusteringTokenizer(); tok.load(tok_path)
        lengths = [len(t) for t in tok._vocab
                   if t not in (b"<|endoftext|>", b"<|pad|>")]
        bpt = float(np.mean(lengths))
        print(f"Clustering avg bytes/token (from vocab): {bpt:.3f}")
        return bpt
    except Exception as e:
        print(f"Could not load tokenizer ({e}), using default 2.8")
        return 2.8

CLS_BPT = get_cls_bpt()

bpe_steps, bpe_vl = zip(*bpe_log) if bpe_log else ([], [])
cls_steps, cls_vl = zip(*cls_log) if cls_log else ([], [])

bpe_bpb = [v / np.log(2) / BPE_BPT for v in bpe_vl]
cls_bpb = [v / np.log(2) / CLS_BPT for v in cls_vl]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(bpe_steps, bpe_vl, "o-", label="BPE", color="steelblue", ms=4)
ax1.plot(cls_steps, cls_vl, "s-", label="Clustering (MinHash+LSH)",
         color="darkorange", ms=4)
ax1.set(xlabel="Step", ylabel="Val Loss (nats, ↓ better)",
        title="Validation Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(bpe_steps, bpe_bpb, "o-", label=f"BPE ({BPE_BPT:.1f} B/tok)",
         color="steelblue", ms=4)
ax2.plot(cls_steps, cls_bpb, "s-", label=f"Clustering ({CLS_BPT:.2f} B/tok)",
         color="darkorange", ms=4)
ax2.set(xlabel="Step", ylabel="Bits per Byte (↓ better)",
        title="BPB — cross-tokenizer fair comparison")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
out = "experiments/out/comparison.png"
import os; os.makedirs("experiments/out", exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out)

In [ ]:
import numpy as np

def summary(name, records, bpt):
    if not records:
        print(f"{name}: no data"); return
    steps, vl = zip(*records)
    bpb = [v / np.log(2) / bpt for v in vl]
    # total tokens consumed at last step (total_batch_size=16384)
    tok = steps[-1] * 16384
    print(f"\n── {name} (avg {bpt:.2f} bytes/token) ──")
    print(f"  Final val loss : {vl[-1]:.4f} nats")
    print(f"  Final BPB      : {bpb[-1]:.4f}")
    print(f"  Steps          : {steps[-1]:,}")
    print(f"  Tokens seen    : {tok/1e6:.1f}M")
    print(f"  Bytes equiv    : {tok*bpt/1e9:.2f} GB")

summary("BPE",        bpe_log, BPE_BPT)
summary("Clustering", cls_log, CLS_BPT)

if bpe_log and cls_log:
    b_bpb = bpe_log[-1][1] / np.log(2) / BPE_BPT
    c_bpb = cls_log[-1][1] / np.log(2) / CLS_BPT
    delta = c_bpb - b_bpb
    winner = "Clustering wins" if delta < 0 else "BPE wins"
    print(f"\nΔ BPB = {delta:+.4f}  →  {winner}")

In [ ]:
import subprocess, sys

# Optional: run eval/tokenizer_metrics.py if it exists
import os
if os.path.exists("eval/tokenizer_metrics.py"):
    subprocess.run([sys.executable, "eval/tokenizer_metrics.py",
                    "--clustering_tokenizer", "data/clustering/tokenizer.bin"],
                   check=False)
else:
    # Inline quick stats
    import sys; sys.path.insert(0, ".")
    from tokenizer.clustering_tokenizer import ClusteringTokenizer
    from tokenizer.bpe_baseline import BPETokenizer

    sample = open("data/raw/tinystories/train_0000.txt", errors="replace").read(50_000)

    bpe = BPETokenizer()
    bpe_toks = bpe.encode(sample)
    bpe_bpt  = len(sample.encode()) / len(bpe_toks)

    cls = ClusteringTokenizer(); cls.load("data/clustering/tokenizer.bin")
    cls_toks = cls.encode(sample)
    cls_bpt  = len(sample.encode()) / len(cls_toks)

    print(f"{'Metric':<28} {'BPE':>10} {'Clustering':>12}")
    print("-" * 52)
    print(f"{'Avg bytes/token':<28} {bpe_bpt:>10.3f} {cls_bpt:>12.3f}")
    print(f"{'Tokens for 50K bytes':<28} {len(bpe_toks):>10,} {len(cls_toks):>12,}")
    words = sample.split()
    print(f"{'Fertility (tok/word)':<28} {len(bpe_toks)/len(words):>10.3f} {len(cls_toks)/len(words):>12.3f}")
    # roundtrip check
    ok = cls.decode_str(cls_toks) == sample
    print(f"{'Roundtrip OK':<28} {'✓':>10} {'✓' if ok else '✗':>12}")